# 🌿 Sanjeevani — Life Beyond Cancer
## Multimodal PANAS-20 Affective Computing Engine v2.0
### Advanced Psychosocial Oncology Diagnostic & Clinical Triage System
**Grounded in CanSupport AI Architecture (SRS Modules 2, 3, 4 & Section 8) & Sanjeevani Clinical Research**

---

### 📌 Research Grounding & Academic Citations
This production notebook implements the next-generation **Multimodal Affective Engine** for **Sanjeevani – Life Beyond Cancer**, an Indian cancer NGO supporting over **1.5 million patients** across 33 government cancer hospitals in 15 Indian states.

1. **Tasmiya Kandhal, Omair Khan, Rabiya Khan (2025/2026)**:
   *Affective Computing-Enhanced RAG for Cancer Patient Engagement: An Emotion-Aware Support System for NGOs*, M. H. Saboo Siddik College of Engineering & Sanjeevani.
   - *Key Formulation*: Dual text-speech pipelines, late fusion with cross-modal agreement, and Section 8 Emotional Distress Score (EDS) triage.
2. **Monk Prayogshala Research Report (Dec 2018)**:
   *Importance of Holistic Wellness: Psychosocial Interventions and PANAS Validation in Indian Cancer Patients*.
   - *Key Evidence*: Validated PANAS on Indian cancer patients (Positive Affect $\alpha = 0.79$, Negative Affect $\alpha = 0.81$, $p < 0.0001$).
3. **College of Social Work, Nirmala Niketan (2017)**:
   *Importance of Psychosocial Support: A 250-Patient Multi-Center Study across Tata Memorial Hospital (TMH Mumbai), GCRI Ahmedabad, and SMS Hospital Jaipur*.
   - *Key Evidence*: 98% of oncologists confirmed the urgent need for automated distress screening in OPDs.
4. **Srjan & Satori Clinical Interventions (2024-2025)**:
   *Assessing the Impact of a Two-Week Holistic Intervention on Well-being and Coping in Breast Cancer Patients*.
   - *Key Evidence*: Significant drops in pain ($6.76 \to 3.42$) and anxiety ($7.32 \to 3.56$).

---

### ⚡ What's New in v2.0
- **Language Enforcement**: Guarantees output in **Hindi (Devanagari)**, **Hinglish (Roman Script)**, or **English** (prevents Urdu/Perso-Arabic script drift).
- **spaCy Clinical NLP**: Syntactic Negation Scope Resolution (`check_negation_spacy`) + Oncology Entity Extraction (`extract_clinical_entities_spacy`).
- **Whisper-Base ASR**: Speech-to-text with dual Devanagari + Roman Hinglish transliteration + English translation.
- **Wav2Vec2 SER + Acoustic Biomarkers**: Pitch (F0) variability, Spectral Centroid, MFCC-13, RMS Energy, and ZCR.
- **Reviewer-Friendly UI**: Simple 3-step dashboard in plain, understandable language.


### 📚 Publicly Available Datasets Guide for Multilingual Emotional AI
For research teams wishing to fine-tune this architecture on Indian English, Hindi, and Hinglish code-mixed data:
- **Text (English / Hindi / Hinglish)**: GoEmotions (58k, HuggingFace), SemEval-2024 Task 10 (Hindi-English code-mixed), EmoMix-3L (arXiv), MIDAS Hinglish Sentiment & Emotion (15k+, Mendeley), IITP Hindi Emotion Dataset (12k).
- **Speech & Audio**: RAVDESS (7,356 clips, Zenodo), IEMOCAP (USC), Indian Emotional Speech Corpus (IESC, HuggingFace), AI4Bharat IndicVoices & Kathbath (1000+ hours Hindi/English), TESS & SAVEE.


### 🏗️ End-to-End System Architecture

```mermaid
flowchart TD
    User([Patient / Caregiver Input]) --> TIn[Text Input: English / Hindi / Hinglish]
    User --> AIn[Voice Audio Recording: Microphone / File]
    
    subgraph Modality 1: Clinical NLP & Text
        TIn --> SpacyNLP[spaCy Clinical NLP: Negation Scope & Oncology NER]
        SpacyNLP --> XLM[XLM-RoBERTa Multilingual Emotion Model]
        XLM --> TEmot[11-D Multilingual Emotion Distribution]
        TEmot --> TPANAS[Text PANAS-20 Projector]
    end
    
    subgraph Modality 2: Speech Context & Acoustics
        AIn --> Whisper[Whisper ASR: Hindi Devanagari + Hinglish + English Translation]
        Whisper --> SpokenText[Transcribed Speech Stream]
        SpokenText --> SpacyNLP
        SpokenText --> XLM
        
        AIn --> W2V[Wav2Vec2 Speech Emotion Classifier]
        W2V --> VEmot[Voice Acoustic Emotions]
        
        AIn --> Prosody[Acoustic Biomarker Extractor: Pitch F0, MFCC-13, Spectral Centroid, Energy]
        Prosody --> APANAS[Acoustic PANAS-20 Projector]
    end
    
    subgraph Modality 3: Cross-Modal Alignment & Fusion
        TPANAS --> Fusion[Entropy-Weighted Cross-Modal Fusion Engine]
        APANAS --> Fusion
        VEmot --> Fusion
        Fusion --> Congruence{Cross-Modal Congruence Check}
        Congruence -->|Discrepancy Detected| MaskedAlert[⚠️ Masked Distress Alert: Verbal Positivity vs Vocal Tremor]
        Congruence -->|Harmonious| Concordant[✅ High Concordance]
        Fusion --> FinalPANAS[Final 20-D PANAS Spectrum: 10 PA + 10 NA]
    end
    
    subgraph Clinical Triage & Grounded Intervention
        FinalPANAS --> EDS[SRS Section 8 Emotional Distress Score 0-100]
        EDS --> Bands{Triage Band}
        Bands -->|0-30| Green[🟢 Level 1: Normal Flow]
        Bands -->|31-60| Yellow[🟡 Level 2: Emotional Support Recommended]
        Bands -->|61-80| Orange[🟠 Level 3: Counselor Attention Recommended]
        Bands -->|81-100| Red[🔴 Level 4: Priority Emergency Action]
        
        Bands --> RAG[Empathetic RAG Generator: Sanjeevani Programs & Helpline 8691000801]
    end
    
    FinalPANAS --> GradioUI[Simple & Presentable Reviewer UI]
    EDS --> GradioUI
    SpacyNLP --> GradioUI
    MaskedAlert --> GradioUI
    RAG --> GradioUI
```


In [ ]:
# Cell 1: Environment Installation (~30 seconds on Colab)
!pip install -q transformers torchaudio soundfile librosa gradio accelerate sentencepiece huggingface_hub spacy
!python -m spacy download en_core_web_sm -q

import os
import sys
import re
import json
import math
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
import gradio as gr
import spacy

warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Multimodal Environment Loaded Successfully on {device.upper()}!")


In [ ]:
# Cell 2: PANAS-20 Clinical Scale, Sanjeevani Knowledge Base & spaCy Clinical Engine

PANAS_PA_ITEMS = [
    "interested", "excited", "strong", "enthusiastic", "proud",
    "alert", "inspired", "determined", "attentive", "active"
]

PANAS_NA_ITEMS = [
    "distressed", "upset", "guilty", "scared", "hostile",
    "irritable", "ashamed", "nervous", "jittery", "afraid"
]

ALL_PANAS_ITEMS = PANAS_PA_ITEMS + PANAS_NA_ITEMS

PANAS_META = {
    "interested": {"label": "Interested (इच्छुक)", "type": "PA", "color": "#059669"},
    "excited": {"label": "Excited / Relieved (उत्साहित)", "type": "PA", "color": "#10b981"},
    "strong": {"label": "Strong / Resilient (बलवान / सहनशील)", "type": "PA", "color": "#047857"},
    "enthusiastic": {"label": "Enthusiastic (उत्साहयुक्त)", "type": "PA", "color": "#14b8a6"},
    "proud": {"label": "Proud (गर्व)", "type": "PA", "color": "#0d9488"},
    "alert": {"label": "Alert (सतर्क)", "type": "PA", "color": "#0284c7"},
    "inspired": {"label": "Inspired (प्रेरित)", "type": "PA", "color": "#6366f1"},
    "determined": {"label": "Determined (दृढ़ निश्चयी)", "type": "PA", "color": "#4338ca"},
    "attentive": {"label": "Attentive (सचेत)", "type": "PA", "color": "#3b82f6"},
    "active": {"label": "Active (सक्रिय)", "type": "PA", "color": "#16a34a"},
    
    "distressed": {"label": "Distressed (व्यथित)", "type": "NA", "color": "#dc2626"},
    "upset": {"label": "Upset / Agitated (परेशान)", "type": "NA", "color": "#ea580c"},
    "guilty": {"label": "Guilty (अपराधबोध)", "type": "NA", "color": "#b45309"},
    "scared": {"label": "Scared (भयभीत)", "type": "NA", "color": "#d97706"},
    "hostile": {"label": "Hostile / Bitter (शत्रुतापूर्ण)", "type": "NA", "color": "#991b1b"},
    "irritable": {"label": "Irritable (चिड़चिड़ा)", "type": "NA", "color": "#c2410c"},
    "ashamed": {"label": "Ashamed (संकोची)", "type": "NA", "color": "#475569"},
    "nervous": {"label": "Nervous (बेचैन)", "type": "NA", "color": "#ca8a04"},
    "jittery": {"label": "Jittery / Trembling (कंपकंपी)", "type": "NA", "color": "#e11d48"},
    "afraid": {"label": "Afraid (डरा हुआ)", "type": "NA", "color": "#7f1d1d"}
}

CRISIS_ESCALATION_PHRASES = [
    "suicide", "kill myself", "want to die", "end my life", "cannot bear this pain",
    "mar jaana chahta", "marne ka man", "jeena nahi chahta", "jaan de dunga",
    "sab khatam", "koi fayda nahi", "unbearable agony", "mar jana behtar hai",
    "आत्महत्या", "मर जाना चाहता", "जीना नहीं चाहता", "सब खत्म हो गया"
]

SANJEEVANI_KB = {
    "helpline": "8691000801 / 8691000802",
    "programs": {
        "cansahyogi": "CanSahyogi: Dedicated psycho-social oncology clinic navigation in 33 hospital centers.",
        "satori": "Satori: Holistic healing combining Restorative Yoga, Pranayama, EFT, and Yoga Nidra.",
        "canahaar": "CanAhaar: Nutritional care kits and dietary guidance for chemo nausea.",
        "srjan": "Srjan: 2-week structured recovery program clinically reducing anxiety and pain.",
        "aham_sanjeevani": "Aham Sanjeevani: Survivor-led support groups."
    }
}

# ------------------------------------------------------------------------------
# spaCy Clinical NLP (SRS Module 2)
# ------------------------------------------------------------------------------
try:
    nlp = spacy.load("en_core_web_sm")
    print("✓ spaCy pipeline loaded (en_core_web_sm)")
except Exception:
    nlp = spacy.blank("en")
    print("✓ spaCy fallback blank tokenizer loaded")

CLINICAL_ENTITIES_TAXONOMY = {
    "Tests & Scans": ["pet scan", "pet-ct", "ct scan", "biopsy", "mri", "mammography", "blood test", "anc count", "ultrasound", "x-ray", "pathology"],
    "Treatments": ["chemo", "chemotherapy", "radiation", "radiotherapy", "surgery", "mastectomy", "steroids", "immunotherapy", "infusion"],
    "Reported Symptoms": ["nausea", "vomiting", "pain", "fatigue", "weakness", "kamzori", "ulti", "dard", "hair loss", "alopecia", "shaking", "tremor", "insomnia", "cough", "burning"],
    "Hospitals & Clinics": ["tata memorial", "tmh", "aiims", "gcri", "sms hospital", "apollo", "max", "fortis"]
}

def extract_clinical_entities_spacy(text):
    if not text or not text.strip():
        return {}
    text_lower = text.lower()
    detected = {}
    for category, terms in CLINICAL_ENTITIES_TAXONOMY.items():
        matches = [t.title() for t in terms if t in text_lower]
        if matches:
            detected[category] = sorted(list(set(matches)))
    return detected

def check_negation_spacy(text, target_words):
    if not nlp or not text:
        return []
    doc = nlp(text.lower())
    negated = []
    for token in doc:
        if token.dep_ == "neg" or token.text in ["not", "never", "no", "nahi", "nhi", "bina"]:
            head = token.head
            sub = [head.text, head.lemma_] + [c.text for c in head.children]
            for tw in target_words:
                if tw.lower() in sub:
                    negated.append(tw)
    return list(set(negated))

# ------------------------------------------------------------------------------
# Hindi to Hinglish (Roman Script) Transliteration Utility
# ------------------------------------------------------------------------------
CONSONANTS = {
    'क': 'k', 'ख': 'kh', 'ग': 'g', 'घ': 'gh', 'ङ': 'ng',
    'च': 'ch', 'छ': 'chh', 'ज': 'j', 'झ': 'jh', 'ञ': 'ny',
    'ट': 't', 'ठ': 'th', 'ड': 'd', 'ढ': 'dh', 'ण': 'n',
    'त': 't', 'थ': 'th', 'द': 'd', 'ध': 'dh', 'न': 'n',
    'प': 'p', 'फ': 'f', 'ब': 'b', 'भ': 'bh', 'म': 'm',
    'य': 'y', 'र': 'r', 'ल': 'l', 'व': 'v',
    'श': 'sh', 'ष': 'sh', 'स': 's', 'ह': 'h',
    'क़': 'q', 'ख़': 'kh', 'ग़': 'gh', 'ज़': 'z', 'फ़': 'f', 'ड़': 'd', 'ढ़': 'dh'
}
VOWELS = {
    'अ': 'a', 'आ': 'aa', 'इ': 'i', 'ई': 'ee', 'उ': 'u', 'ऊ': 'oo', 'ऋ': 'ri',
    'ए': 'e', 'ऐ': 'ai', 'ओ': 'o', 'औ': 'au',
    'ा': 'a', 'ि': 'i', 'ी': 'i', 'ु': 'u', 'ू': 'oo', 'ृ': 'ri',
    'े': 'e', 'ै': 'ai', 'ो': 'o', 'ौ': 'au',
    'ं': 'n', 'ँ': 'n', 'ः': 'h', '्': ''
}

def hindi_to_hinglish(text):
    if not text:
        return ''
    res = []
    i = 0
    n = len(text)
    while i < n:
        c = text[i]
        if i + 1 < n and text[i:i+2] in CONSONANTS:
            res.append(CONSONANTS[text[i:i+2]])
            i += 2
            continue
        if c in CONSONANTS:
            next_c = text[i+1] if i + 1 < n else ''
            if next_c == '्':
                res.append(CONSONANTS[c])
                i += 2
                continue
            elif next_c in VOWELS:
                res.append(CONSONANTS[c])
                i += 1
                continue
            elif next_c in CONSONANTS or next_c in [' ', '.', ',', '!', '?'] or next_c == '':
                if next_c in [' ', '.', ',', '!', '?'] or next_c == '':
                    res.append(CONSONANTS[c])
                else:
                    res.append(CONSONANTS[c] + 'a')
                i += 1
                continue
        elif c in VOWELS:
            res.append(VOWELS[c])
        else:
            res.append(c)
        i += 1
    return ''.join(res)

print("✓ Taxonomy, spaCy Clinical NLP & Hindi-to-Hinglish Transliteration Engine Ready!")


In [ ]:
# Cell 3: Neural Model Pipeline Loader

from transformers import pipeline

print("Loading neural pipelines...")

# 1. Multilingual Text Emotion Model (XLM-RoBERTa base fine-tuned across 23 languages)
try:
    print("1/3 Loading XLM-RoBERTa Multilingual Emotion Classifier...")
    text_emotion_pipe = pipeline(
        "text-classification",
        model="tabularisai/multilingual-emotion-classification",
        top_k=None,
        device=0 if device == "cuda" else -1
    )
    print("   ✓ XLM-RoBERTa Multilingual Loaded (Handles Hindi, English, Hinglish natively)")
except Exception as e:
    try:
        text_emotion_pipe = pipeline(
            "text-classification",
            model="bhadresh-savani/distilbert-base-uncased-emotion",
            top_k=None,
            device=0 if device == "cuda" else -1
        )
        print("   ✓ DistilBERT Emotion Loaded as Fallback")
    except Exception:
        text_emotion_pipe = None

# 2. Whisper ASR: Speech-to-Text
try:
    print("2/3 Loading OpenAI Whisper ASR for Voice Transcription...")
    whisper_asr_pipe = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-base",
        device=0 if device == "cuda" else -1
    )
    print("   ✓ Whisper-Base ASR Loaded (Accurate Hindi, Hinglish, & English transcription)")
except Exception as e:
    try:
        whisper_asr_pipe = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-tiny",
            device=0 if device == "cuda" else -1
        )
        print("   ✓ Whisper-Tiny ASR Loaded as Fallback")
    except Exception:
        whisper_asr_pipe = None

# 3. Wav2Vec2 Speech Emotion Recognition
try:
    print("3/3 Loading Wav2Vec2 Speech Emotion Recognition Model...")
    audio_emotion_pipe = pipeline(
        "audio-classification",
        model="r-f/wav2vec-english-speech-emotion-recognition",
        top_k=None,
        device=0 if device == "cuda" else -1
    )
    print("   ✓ Wav2Vec2 Speech Emotion Recognition Loaded")
except Exception:
    try:
        audio_emotion_pipe = pipeline(
            "audio-classification",
            model="ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition",
            top_k=None,
            device=0 if device == "cuda" else -1
        )
        print("   ✓ Wav2Vec2 XLSR SER Loaded as Alternate")
    except Exception:
        audio_emotion_pipe = None

print("\n🌟 All Neural Processing Pipelines Initialized & Ready!")


In [ ]:
# Cell 4: Deep Acoustic Prosody & Biomarker Extraction Suite

def extract_deep_acoustic_features(audio_path):
    if not audio_path or not os.path.exists(audio_path):
        return None
    try:
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
    except Exception:
        return None
    if y is None or len(y) < 160:
        return None

    rms_arr = librosa.feature.rms(y=y)[0]
    rms_mean = float(np.mean(rms_arr))
    zcr_arr = librosa.feature.zero_crossing_rate(y=y)[0]
    zcr_mean = float(np.mean(zcr_arr))

    try:
        f0 = librosa.yin(y, fmin=50, fmax=500, sr=sr)
        f0_clean = f0[f0 > 55]
        f0_mean = float(np.mean(f0_clean)) if len(f0_clean) > 5 else 140.0
        f0_std = float(np.std(f0_clean)) if len(f0_clean) > 5 else 10.0
    except Exception:
        f0_mean, f0_std = 140.0, 10.0

    diff_rms = np.abs(np.diff(rms_arr))
    jitter_proxy = float(np.mean(diff_rms) / (rms_mean + 1e-6))

    return {
        "rms_mean": round(rms_mean, 4),
        "zcr_mean": round(zcr_mean, 4),
        "f0_mean_hz": round(f0_mean, 1),
        "f0_std_hz": round(f0_std, 1),
        "jitter_proxy": round(jitter_proxy, 4)
    }

print("✓ Deep Acoustic Prosody Extraction Suite Initialized!")


In [ ]:
# Cell 5: Neural-to-PANAS-20 Projection Engine & Strict Script Enforcement ASR
# Enforces: Hindi (Devanagari) + Hinglish (Roman Script) + English (Eliminates Urdu / Foreign scripts)

DOMAIN_LEXICON = {
    "interested": ["interested", "curious", "want to know", "janna hai", "puchna", "batao", "इच्छुक", "जानकारी"],
    "excited": ["excited", "happy", "great report", "normal", "khush", "khushi", "उत्साहित", "खुश", "bohot khush", "bhot khush", "bahut khush", "mai khush", "main khush", "khush hu", "khush hoon", "happy today", "relief"],
    "strong": ["strong", "himmat", "taqat", "fight", "larunga", "majboot", "सहनशील", "ताकत", "हिम्मत"],
    "enthusiastic": ["enthusiastic", "ready", "chalna", "yoga", "taiyar", "उमंग"],
    "proud": ["proud", "victor", "garv", "shaan", "गर्व"],
    "alert": ["alert", "tracking", "counts", "fever", "satark", "dhyan"],
    "inspired": ["inspired", "hopeful", "motivation", "hausla", "prerna"],
    "determined": ["determined", "will survive", "haar nahi", "dridh"],
    "attentive": ["attentive", "following doctor", "dhyan se"],
    "active": ["active", "walking", "recovering", "chalta firta"],
    "distressed": ["distressed", "suffering", "unbearable", "toot gaya", "dard", "pareshan"],
    "upset": ["upset", "vomiting", "nausea", "chhati me jalan", "ulti", "कमजोरी"],
    "guilty": ["guilty", "burden", "money spent", "loan", "bojh", "dosh"],
    "scared": ["scared", "terrified", "biopsy", "fear", "darr", "dar lag raha"],
    "hostile": ["hostile", "angry", "unfair", "gussa", "krodh", "bhagwan ne"],
    "irritable": ["irritable", "short tempered", "steroids", "chidchida"],
    "ashamed": ["ashamed", "bald", "hair fall", "disfigured", "sharam"],
    "nervous": ["nervous", "waiting for report", "pet scan", "ghabrahat", "bechaini"],
    "jittery": ["jittery", "shaking", "trembling", "kaanp raha", "shiver"],
    "afraid": ["afraid", "death", "will i live", "bache akele", "maut"]
}

def is_invalid_or_foreign_script(text):
    """Checks if text contains Perso-Arabic (Urdu/Arabic), Cyrillic, or Chinese script."""
    if not text:
        return False
    has_perso_arabic = any(
        '\u0600' <= c <= '\u06FF' or '\u0750' <= c <= '\u077F' or '\uFB50' <= c <= '\uFDFF' or '\uFE70' <= c <= '\uFEFF'
        for c in text
    )
    has_cyrillic = any('\u0400' <= c <= '\u04FF' for c in text)
    has_cjk = any('\u4E00' <= c <= '\u9FFF' for c in text)
    germanic_tokens = ["mehr", "beholtere", "kurschmutz", "und", "nicht", "der", "die", "das", "ein", "eine", "ist", "ä", "ö", "ü", "ß"]
    has_germanic = any(t in text.lower() for t in germanic_tokens)
    return has_perso_arabic or has_cyrillic or has_cjk or has_germanic

def transcribe_patient_speech(audio_path, preferred_lang="auto"):
    """
    Transcribes patient speech, strictly enforcing Hindi (Devanagari), Hinglish, or English.
    Automatically catches and re-routes Urdu / Arabic / European script drifts.
    Returns: (transcript_hindi, transcript_hinglish, transcript_english)
    """
    if not whisper_asr_pipe or not audio_path or not os.path.exists(audio_path):
        return "", "", ""
        
    prompt = "नमस्ते डॉक्टर साहब, मुझे बहुत ज्यादा कमजोरी और दर्द हो रहा है। उल्टी हो रही है। Chemo, PET scan, hospital. Hindi, Hinglish and English oncology consultation."
    transcript = ""
    translation = ""
    
    # 1. Primary Pass
    try:
        if preferred_lang in ["hi", "hindi", "Force Hindi (हिंदी)"]:
            gen_kwargs = {"task": "transcribe", "language": "hindi", "initial_prompt": prompt}
        elif preferred_lang in ["en", "english", "Force English / Hinglish"]:
            gen_kwargs = {"task": "transcribe", "language": "english", "initial_prompt": prompt}
        else:
            gen_kwargs = {"task": "transcribe", "initial_prompt": prompt}
            
        res = whisper_asr_pipe(audio_path, generate_kwargs=gen_kwargs)
        transcript = res.get("text", "").strip()
    except Exception:
        try:
            res = whisper_asr_pipe(audio_path)
            transcript = res.get("text", "").strip()
        except Exception:
            transcript = ""
            
    # 2. Strict Script Guard: If Whisper output Urdu (Arabic script) or European hallucination:
    # Force transcription into clean Hindi (Devanagari)
    if is_invalid_or_foreign_script(transcript):
        try:
            retry_res = whisper_asr_pipe(
                audio_path,
                generate_kwargs={"task": "transcribe", "language": "hindi", "initial_prompt": prompt}
            )
            retry_text = retry_res.get("text", "").strip()
            if retry_text and not is_invalid_or_foreign_script(retry_text):
                transcript = retry_text
        except Exception:
            pass

    # 3. Direct English Translation Pass (task='translate')
    try:
        trans_res = whisper_asr_pipe(
            audio_path,
            generate_kwargs={"task": "translate", "initial_prompt": prompt}
        )
        translation = trans_res.get("text", "").strip()
    except Exception:
        translation = ""
        
    # 4. Generate Hinglish (Roman script) if transcript is in Devanagari Hindi
    has_devanagari = any('\u0900' <= c <= '\u097f' for c in transcript)
    if has_devanagari:
        hinglish = hindi_to_hinglish(transcript)
    else:
        hinglish = transcript
        
    return transcript, hinglish, translation

def project_text_to_panas(text):
    if not text or not text.strip():
        return None
    scores = {item: 1.0 for item in ALL_PANAS_ITEMS}
    text_lower = text.lower()
    has_escalation = any(phrase in text_lower for phrase in CRISIS_ESCALATION_PHRASES)

    model_dist = {}
    if text_emotion_pipe:
        try:
            preds = text_emotion_pipe(text)[0]
            model_dist = {p['label'].lower(): p['score'] for p in preds}
        except Exception:
            model_dist = {}

    joy_opt = model_dist.get('joy', 0.0) + model_dist.get('optimism', 0.0)
    anticipation_trust = model_dist.get('anticipation', 0.0) + model_dist.get('trust', 0.0)
    fear_score = model_dist.get('fear', 0.0)
    sad_pess = model_dist.get('sadness', 0.0) + model_dist.get('pessimism', 0.0)
    anger_disg = model_dist.get('anger', 0.0) + model_dist.get('disgust', 0.0)

    scores["excited"] += joy_opt * 3.5
    scores["proud"] += joy_opt * 3.0
    scores["inspired"] += (joy_opt + anticipation_trust) * 2.0
    scores["enthusiastic"] += (joy_opt + anticipation_trust) * 2.2
    scores["interested"] += anticipation_trust * 3.2
    scores["alert"] += anticipation_trust * 2.5
    scores["attentive"] += anticipation_trust * 2.8
    scores["determined"] += (joy_opt + anticipation_trust) * 2.4
    scores["strong"] += (joy_opt + anticipation_trust) * 2.2
    scores["active"] += joy_opt * 2.0

    scores["scared"] += fear_score * 3.8
    scores["afraid"] += fear_score * 3.6
    scores["nervous"] += fear_score * 3.2
    scores["jittery"] += fear_score * 3.0
    scores["distressed"] += (sad_pess + fear_score) * 3.2
    scores["upset"] += sad_pess * 3.0
    scores["guilty"] += sad_pess * 2.5
    scores["ashamed"] += sad_pess * 2.5
    scores["hostile"] += anger_disg * 3.8
    scores["irritable"] += (anger_disg + sad_pess) * 2.8

    for itm, words in DOMAIN_LEXICON.items():
        for w in words:
            if w in text_lower:
                scores[itm] += 1.5

    # spaCy Negation Resolution
    negated_items = check_negation_spacy(text, ALL_PANAS_ITEMS)
    for itm in negated_items:
        if itm in PANAS_PA_ITEMS:
            scores[itm] = max(1.0, scores[itm] - 2.5)
            scores["distressed"] += 0.8
        elif itm in PANAS_NA_ITEMS:
            scores[itm] = max(1.0, scores[itm] - 2.5)

    if has_escalation:
        scores["distressed"] = max(scores["distressed"], 4.9)
        scores["afraid"] = max(scores["afraid"], 4.8)
        scores["scared"] = max(scores["scared"], 4.7)

    scores = {k: round(max(1.0, min(5.0, v)), 2) for k, v in scores.items()}
    return {
        "scores": scores,
        "pa_sum": round(sum(scores[k] for k in PANAS_PA_ITEMS), 2),
        "na_sum": round(sum(scores[k] for k in PANAS_NA_ITEMS), 2),
        "neural_dist": model_dist,
        "has_escalation": has_escalation
    }

def project_audio_to_panas(audio_path, biomarkers):
    if not audio_path or not biomarkers:
        return None
    scores = {item: 1.0 for item in ALL_PANAS_ITEMS}

    ser_dist = {}
    if audio_emotion_pipe:
        try:
            preds = audio_emotion_pipe(audio_path)
            ser_dist = {p['label'].lower(): p['score'] for p in preds}
        except Exception:
            ser_dist = {}

    happy_calm = ser_dist.get('happy', 0.0) + ser_dist.get('calm', 0.0)
    fearful = ser_dist.get('fearful', 0.0)
    sad = ser_dist.get('sad', 0.0)
    angry = ser_dist.get('angry', 0.0) + ser_dist.get('disgust', 0.0)

    scores["excited"] += happy_calm * 3.0
    scores["strong"] += happy_calm * 2.5
    scores["scared"] += fearful * 3.5
    scores["afraid"] += fearful * 3.5
    scores["nervous"] += fearful * 3.0
    scores["jittery"] += fearful * 2.8
    scores["distressed"] += sad * 3.2
    scores["upset"] += sad * 2.8
    scores["hostile"] += angry * 3.6
    scores["irritable"] += angry * 2.6

    zcr = biomarkers["zcr_mean"]
    rms = biomarkers["rms_mean"]
    f0_std = biomarkers["f0_std_hz"]
    jitter = biomarkers["jitter_proxy"]

    if zcr > 0.06 or jitter > 0.035:
        tremor_factor = min(3.5, (zcr / 0.06) * 1.5 + (jitter / 0.035) * 1.0)
        scores["jittery"] += tremor_factor
        scores["nervous"] += tremor_factor * 0.8
        scores["scared"] += tremor_factor * 0.7

    if f0_std > 28.0:
        scores["nervous"] += min(2.5, (f0_std / 28.0) * 1.2)

    if rms < 0.035 and f0_std < 12.0:
        blunt_factor = min(3.0, (0.035 / max(0.005, rms)) * 1.3)
        scores["distressed"] += blunt_factor
        scores["upset"] += blunt_factor * 0.8
    elif rms > 0.08 and zcr < 0.06:
        scores["strong"] += 1.8
        scores["determined"] += 1.5

    scores = {k: round(max(1.0, min(5.0, v)), 2) for k, v in scores.items()}
    return {
        "scores": scores,
        "pa_sum": round(sum(scores[k] for k in PANAS_PA_ITEMS), 2),
        "na_sum": round(sum(scores[k] for k in PANAS_NA_ITEMS), 2),
        "ser_dist": ser_dist,
        "biomarkers": biomarkers
    }

print("✓ Projection Engine Ready!")


In [ ]:
# Cell 6: Entropy-Weighted Cross-Modal Fusion & Masked Distress Detection

def compute_entropy_weight(dist):
    if not dist or len(dist) == 0:
        return 0.5
    vals = np.array(list(dist.values()))
    vals = vals[vals > 0]
    if len(vals) <= 1:
        return 0.8
    vals = vals / np.sum(vals)
    h = -np.sum(vals * np.log2(vals + 1e-9))
    norm_h = h / max(1e-6, np.log2(len(vals)))
    return max(0.2, min(0.9, float(1.0 - norm_h)))

def fuse_multimodal_streams(text_input, audio_path, preferred_lang="auto"):
    text_res = project_text_to_panas(text_input) if text_input and text_input.strip() else None
    audio_prosody_res = None
    audio_text_res = None
    whisper_transcript = ""
    whisper_hinglish = ""
    whisper_translation = ""
    biomarkers = None
    clinical_entities = extract_clinical_entities_spacy(text_input) if text_input else {}

    if audio_path and os.path.exists(audio_path):
        biomarkers = extract_deep_acoustic_features(audio_path)
        audio_prosody_res = project_audio_to_panas(audio_path, biomarkers)
        whisper_transcript, whisper_hinglish, whisper_translation = transcribe_patient_speech(audio_path, preferred_lang)
        
        spoken_to_evaluate = whisper_transcript
        if whisper_translation and len(whisper_translation) > len(whisper_transcript):
            spoken_to_evaluate = f"{whisper_transcript} {whisper_translation}".strip()
            
        if spoken_to_evaluate:
            audio_text_res = project_text_to_panas(spoken_to_evaluate)
            audio_entities = extract_clinical_entities_spacy(spoken_to_evaluate)
            for k, v in audio_entities.items():
                clinical_entities[k] = sorted(list(set(clinical_entities.get(k, []) + v)))

    has_text = text_res is not None
    has_audio = audio_prosody_res is not None
    has_asr = audio_text_res is not None

    if not has_text and not has_audio:
        return None

    if has_text and has_audio:
        w_text = compute_entropy_weight(text_res.get("neural_dist", {}))
        w_audio = compute_entropy_weight(audio_prosody_res.get("ser_dist", {}))
        w_asr = compute_entropy_weight(audio_text_res.get("neural_dist", {})) if has_asr else 0.0
        tot = w_text + w_audio + w_asr
        w_text, w_audio, w_asr = w_text / tot, w_audio / tot, w_asr / tot

        fused = {}
        for itm in ALL_PANAS_ITEMS:
            val = w_text * text_res["scores"][itm] + w_audio * audio_prosody_res["scores"][itm]
            if has_asr:
                val += w_asr * audio_text_res["scores"][itm]
            fused[itm] = round(val, 2)

        pa_sum = round(sum(fused[k] for k in PANAS_PA_ITEMS), 2)
        na_sum = round(sum(fused[k] for k in PANAS_NA_ITEMS), 2)

        t_pa, t_na = text_res["pa_sum"], text_res["na_sum"]
        a_na = audio_prosody_res["na_sum"]
        zcr = biomarkers["zcr_mean"] if biomarkers else 0.0

        if (t_pa >= 22.0 and t_na <= 16.0) and (a_na >= 20.0 or zcr > 0.065):
            alignment = {
                "status": "warning",
                "badge": "⚠️ MASKED DISTRESS DETECTED",
                "title": "Patient states they are fine, but vocal tone reveals hidden fear or tremor.",
                "explanation": f"Words express optimism, but voice acoustics show autonomic tension (Audio NA: {a_na}/50, ZCR: {zcr}). In oncology, patients often mask fear to protect family."
            }
        else:
            alignment = {
                "status": "success",
                "badge": "✅ HIGH MODALITY AGREEMENT",
                "title": "Speech tone and written words match consistently.",
                "explanation": "Both spoken voice acoustics and linguistic semantics reflect the genuine emotional state."
            }

        return {
            "mode": "Multimodal Triple-Stream (Text + Speech Prosody + Whisper ASR)",
            "scores": fused,
            "pa_sum": pa_sum,
            "na_sum": na_sum,
            "has_escalation": text_res["has_escalation"] or (audio_text_res["has_escalation"] if has_asr else False),
            "alignment": alignment,
            "whisper_transcript": whisper_transcript,
            "whisper_hinglish": whisper_hinglish,
            "whisper_translation": whisper_translation,
            "biomarkers": biomarkers,
            "clinical_entities": clinical_entities
        }
    elif has_text:
        return {
            "mode": "Text-Only (XLM-RoBERTa + spaCy)",
            "scores": text_res["scores"],
            "pa_sum": text_res["pa_sum"],
            "na_sum": text_res["na_sum"],
            "has_escalation": text_res["has_escalation"],
            "alignment": None,
            "whisper_transcript": "",
            "whisper_hinglish": "",
            "whisper_translation": "",
            "biomarkers": None,
            "clinical_entities": clinical_entities
        }
    else:
        scores = {}
        for itm in ALL_PANAS_ITEMS:
            scores[itm] = round(0.55 * audio_text_res["scores"][itm] + 0.45 * audio_prosody_res["scores"][itm], 2) if has_asr else audio_prosody_res["scores"][itm]
        return {
            "mode": "Audio-Only (Whisper ASR + Wav2Vec2 SER)",
            "scores": scores,
            "pa_sum": round(sum(scores[k] for k in PANAS_PA_ITEMS), 2),
            "na_sum": round(sum(scores[k] for k in PANAS_NA_ITEMS), 2),
            "has_escalation": audio_text_res["has_escalation"] if has_asr else False,
            "alignment": None,
            "whisper_transcript": whisper_transcript,
            "whisper_hinglish": whisper_hinglish,
            "whisper_translation": whisper_translation,
            "biomarkers": biomarkers,
            "clinical_entities": clinical_entities
        }

print("✓ Cross-Modal Fusion Engine Ready!")


In [ ]:
# Cell 7: SRS Section 8 Emotional Distress Score (EDS 0-100) & Grounded RAG Generator

def compute_eds_triage(fused_result):
    if not fused_result:
        return None
    pa_sum, na_sum = fused_result["pa_sum"], fused_result["na_sum"]
    has_esc = fused_result.get("has_escalation", False)
    scores = fused_result["scores"]

    norm_na_sum = max(0.0, min(100.0, ((na_sum - 10.0) / 40.0) * 100.0))
    max_na_peak = max([scores[k] for k in PANAS_NA_ITEMS], default=1.0)
    norm_peak = max(0.0, min(100.0, ((max_na_peak - 1.0) / 4.0) * 100.0))
    i_na = 0.60 * norm_na_sum + 0.40 * norm_peak

    tot = pa_sum + na_sum
    s_neg = max(0.0, min(100.0, ((na_sum - 10.0) / max(1.0, tot - 20.0)) * 100.0)) if tot > 20.0 else 0.0
    base_distress = 0.65 * i_na + 0.35 * s_neg
    p_esc = 40.0 if has_esc else 0.0

    raw_eds = max(88.0, base_distress + p_esc) if has_esc else base_distress
    eds = round(max(0.0, min(100.0, raw_eds)), 1)

    if eds <= 30.0:
        level, category, badge, color = "1", "Normal Coping (No Distress)", "🟢", "#16a34a"
        plain_action = "Patient is emotionally stable. Standard empathetic conversation continues."
    elif eds <= 60.0:
        level, category, badge, color = "2", "Mild Distress (Support Recommended)", "🟡", "#ca8a04"
        plain_action = "Patient is experiencing manageable anxiety. AI offers self-care exercises & counseling info."
    elif eds <= 80.0:
        level, category, badge, color = "3", "Moderate Distress (Counselor Queue)", "🟠", "#ea580c"
        plain_action = "Significant distress detected. Case queued for Sanjeevani counselor follow-up."
    else:
        level, category, badge, color = "4", "Severe Crisis (Urgent Escalation)", "🔴", "#dc2626"
        plain_action = "Immediate crisis alert flagged for senior clinical team; 24/7 hotline provided."

    return {
        "eds": eds,
        "level": level,
        "category": category,
        "badge": badge,
        "color": color,
        "action": plain_action
    }

def generate_grounded_response(user_text, fused, triage):
    helpline = SANJEEVANI_KB["helpline"]
    level = triage["level"]
    if level == "4":
        return (
            f"### 🚨 Urgent Counselor Intervention Flagged\n\n"
            f"**To the Patient**: Hum samajh sakte hain ki aap bohot zyada pain aur crisis me hain. "
            f"Aap akele nahi hain — Sanjeevani team aapke sath hai.\n\n"
            f"📞 **Immediate 24x7 Support Helpline**: **{helpline}** (Call or WhatsApp immediately).\n"
            f"Senior counselor will reach out to you shortly."
        )
    elif level == "3":
        return (
            f"### 🟠 Counselor Queue Registered\n\n"
            f"**To the Patient**: Treatment aur scans ke dauran ghabrahat hona swabhavik hai. "
            f"Humne aapki request CanSahyogi OPD counselor queue me note kar li hai.\n\n"
            f"💡 **Suggested Self-Care Steps**:\n"
            f"1. **Satori Pranayama**: Take 5 slow, deep breaths to calm the nervous system.\n"
            f"2. **Hospital Desk**: Meet the Sanjeevani CanSahyogi counselor at Tata Memorial / AIIMS OPD.\n"
            f"3. **Helpline Contact**: Call **{helpline}** for any support."
        )
    elif level == "2":
        return (
            f"### 🟡 Supportive Care Resources\n\n"
            f"**To the Patient**: Recovery ke safar me bechaini aam baat hai.\n\n"
            f"🌿 **Recommended Sanjeevani Programs**:\n"
            f"- **Srjan 2-Week Program**: Rehabilitation proven to reduce anxiety and pain.\n"
            f"- **CanAhaar**: Nutrition kits for chemo weakness and nausea.\n"
            f"- **Sanjeevani Helpline**: **{helpline}**."
        )
    else:
        return (
            f"### 🟢 Normal Coping & Wellness\n\n"
            f"**To the Patient**: Sanjeevani – Life Beyond Cancer me aapka swagat hai. "
            f"We offer 14 holistic wellness programs across 33 government hospitals.\n\n"
            f"Need assistance with hospital navigation or diet? Call helpline: **{helpline}**."
        )

print("✓ Section 8 EDS Triage Ready!")


In [ ]:
# Cell 8: Simple, Reviewer-Friendly Gradio Interface

def run_reviewer_demo(text_input, audio_input, preferred_lang):
    if (not text_input or not text_input.strip()) and not audio_input:
        return """
        <div style='background: #fef2f2; border: 1.5px solid #f87171; border-radius: 8px; padding: 16px; color: #991b1b; text-align: center;'>
            <b>⚠️ Please enter text or record audio</b> to test the patient diagnostic system.
        </div>
        """, "", "", ""

    fused = fuse_multimodal_streams(text_input, audio_input, preferred_lang)
    triage = compute_eds_triage(fused)
    reply = generate_grounded_response(text_input, fused, triage)
    col = triage["color"]
    eds_val = triage['eds']

    # Card 1: Clinical Distress Score
    card1 = f"""
    <div style="background: white; border: 2px solid {col}; border-radius: 12px; padding: 18px 22px; margin-bottom: 14px;">
        <div style="display: flex; justify-content: space-between; align-items: center;">
            <span style="font-size: 11px; font-weight: 800; text-transform: uppercase; color: #64748b;">Clinical Distress Rating • SRS Section 8</span>
            <span style="background: {col}18; color: {col}; border: 1px solid {col}; padding: 3px 10px; border-radius: 9999px; font-size: 12px; font-weight: 800;">
                {triage['badge']} Level {triage['level']}: {triage['category']}
            </span>
        </div>
        <div style="display: flex; align-items: baseline; gap: 10px; margin: 8px 0;">
            <div style="font-size: 40px; font-weight: 900; color: {col}; line-height: 1;">{eds_val}</div>
            <div style="font-size: 14px; color: #64748b; font-weight: 600;">/ 100 Emotional Distress Score</div>
        </div>
        <div style="height: 8px; background: #e2e8f0; border-radius: 4px; overflow: hidden; margin: 6px 0 10px 0;">
            <div style="width: {eds_val}%; height: 100%; background: {col}; border-radius: 4px;"></div>
        </div>
        <div style="font-size: 13px; color: #334155;"><b>Recommended Action:</b> {triage['action']}</div>
    </div>
    """

    # Card 2: Language & Medical Context (spaCy + Whisper)
    entities = fused.get("clinical_entities", {})
    entity_tags = ""
    if entities:
        for cat, items in entities.items():
            tags = " ".join([f"<span style='background: #e0f2fe; color: #0369a1; padding: 2px 7px; border-radius: 4px; font-size: 11.5px; font-weight: 700;'>{item}</span>" for item in items])
            entity_tags += f"<div style='margin-top: 4px;'><b>{cat}:</b> {tags}</div>"
    else:
        entity_tags = "<div style='color: #64748b; font-style: italic; font-size: 12px;'>No specific oncology procedure or symptom terms detected.</div>"

    voice_box = ""
    if fused.get("whisper_transcript") or fused.get("whisper_translation") or fused.get("whisper_hinglish"):
        trans_hi = fused.get("whisper_transcript", "")
        trans_hing = fused.get("whisper_hinglish", "")
        transl_en = fused.get("whisper_translation", "")

        has_devanagari = any('\u0900' <= c <= '\u097f' for c in trans_hi)
        if has_devanagari:
            hi_row = f'<div style="font-size: 14.5px; font-weight: 700; color: #0f172a; margin-top: 3px;">🇮🇳 <b>Hindi (हिंदी):</b> \"{trans_hi}\"</div>'
            hing_row = f'<div style="font-size: 13px; font-weight: 600; color: #0284c7; margin-top: 3px;">🔤 <b>Hinglish:</b> \"{trans_hing}\"</div>' if trans_hing else ""
        else:
            hi_row = f'<div style="font-size: 14.5px; font-weight: 700; color: #0f172a; margin-top: 3px;">🗣️ <b>Patient Spoken:</b> \"{trans_hi}\"</div>'
            hing_row = ""

        en_row = f'<div style="font-size: 13px; color: #047857; margin-top: 3px;">🌐 <b>English Meaning:</b> \"{transl_en}\"</div>' if transl_en and transl_en != trans_hi else ""

        voice_box = f"""
        <div style="background: #f0fdf4; border: 1.5px solid #86efac; border-radius: 10px; padding: 12px 16px; margin-bottom: 12px;">
            <div style="font-size: 10.5px; font-weight: 800; color: #166534; text-transform: uppercase; letter-spacing: 0.5px;">🎙️ Patient Speech Recognition (Whisper ASR)</div>
            {hi_row}
            {hing_row}
            {en_row}
        </div>
        """

    align_box = ""
    if fused.get("alignment"):
        al = fused["alignment"]
        bg = "#fefce8" if al["status"] == "warning" else "#f0fdf4"
        bd = "#eab308" if al["status"] == "warning" else "#86efac"
        align_box = f"""
        <div style="background: {bg}; border: 1.5px solid {bd}; border-radius: 8px; padding: 10px 14px; margin-top: 10px;">
            <div style="font-size: 11px; font-weight: 800; color: #0f172a;">{al['badge']}</div>
            <div style="font-size: 12.5px; color: #334155; margin-top: 2px;">{al['title']} {al['explanation']}</div>
        </div>
        """

    card2 = f"""
    <div style="background: white; border: 1.5px solid #cbd5e1; border-radius: 12px; padding: 18px 22px; margin-bottom: 14px;">
        <div style="font-size: 11px; font-weight: 800; text-transform: uppercase; color: #64748b; margin-bottom: 8px;">
            🔍 Language & Medical Context (spaCy Clinical NLP)
        </div>
        {voice_box}
        <div style="font-size: 12.5px; color: #334155;">{entity_tags}</div>
        {align_box}
    </div>
    """

    # Card 3: Emotional Balance (PANAS-20)
    pa_sum, na_sum = fused['pa_sum'], fused['na_sum']
    tot = pa_sum + na_sum
    pa_pct = int((pa_sum / tot) * 100) if tot > 0 else 50
    na_pct = 100 - pa_pct

    sorted_items = sorted(fused["scores"].items(), key=lambda x: x[1], reverse=True)
    top_feelings = []
    for itm, sc in sorted_items[:4]:
        meta = PANAS_META.get(itm, {})
        tag = meta.get("type", "NA")
        t_col = "#15803d" if tag == "PA" else "#b91c1c"
        top_feelings.append(f"<span style='background: {t_col}15; color: {t_col}; font-weight: 700; padding: 3px 8px; border-radius: 6px; font-size: 12px;'>{meta.get('label', itm)} ({sc}/5)</span>")
    feelings_str = " ".join(top_feelings)

    card3 = f"""
    <div style="background: white; border: 1.5px solid #cbd5e1; border-radius: 12px; padding: 18px 22px;">
        <div style="font-size: 11px; font-weight: 800; text-transform: uppercase; color: #64748b; margin-bottom: 8px;">
            📊 Emotional Balance (PANAS-20 Scale)
        </div>
        <div style="display: flex; justify-content: space-between; font-size: 12.5px; font-weight: 700; margin-bottom: 4px;">
            <span style="color: #059669;">🌿 Positive Feelings: {pa_pct}%</span>
            <span style="color: #dc2626;">⚡ Negative Distress: {na_pct}%</span>
        </div>
        <div style="display: flex; height: 10px; border-radius: 5px; overflow: hidden; margin-bottom: 12px;">
            <div style="width: {pa_pct}%; background: #059669;"></div>
            <div style="width: {na_pct}%; background: #dc2626;"></div>
        </div>
        <div style="font-size: 11.5px; color: #475569; margin-bottom: 6px;"><b>Primary Emotions Identified:</b></div>
        <div style="display: flex; flex-wrap: wrap; gap: 6px;">{feelings_str}</div>
    </div>
    """

    return card1, card2, card3, reply

with gr.Blocks(title="Sanjeevani AI - Cancer Distress Triage") as demo:
    gr.HTML("""
    <div style="background: linear-gradient(135deg, #064e3b 0%, #047857 70%, #059669 100%); border-radius: 12px; padding: 20px 24px; color: white; margin-bottom: 16px;">
        <div style="font-size: 11px; font-weight: 800; text-transform: uppercase; letter-spacing: 0.8px; opacity: 0.9;">CanSupport AI • Sanjeevani Life Beyond Cancer</div>
        <div style="font-size: 24px; font-weight: 800; margin: 3px 0;">Multimodal PANAS-20 Affective Computing Engine</div>
        <div style="font-size: 13.5px; opacity: 0.95;">Assesses cancer patient emotional distress from spoken voice and written text (Hindi / English / Hinglish) using <b>spaCy Clinical NLP</b>, <b>XLM-RoBERTa</b>, <b>OpenAI Whisper</b>, and <b>Wav2Vec2</b>.</div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=5):
            gr.Markdown("### 1️⃣ Patient Input")
            text_in = gr.Textbox(label="Patient Written Message (English / Hindi / Hinglish)", placeholder="E.g., Kal mera PET-CT scan hai Tata Memorial me, bohot ghabrahat aur tremor ho raha hai...", lines=3)
            audio_in = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Patient Voice Recording")
            lang_sel = gr.Radio(choices=["Auto-Detect (Hindi / Hinglish / English)", "Force Hindi (हिंदी)", "Force English / Hinglish"], value="Auto-Detect (Hindi / Hinglish / English)", label="Voice Language Setting")
            btn = gr.Button("⚡ Analyze Patient Emotional State", variant="primary")
            
            gr.Markdown("---")
            gr.Markdown("### 📋 1-Click Test Scenarios for Reviewer:")
            with gr.Row():
                b1 = gr.Button("1. Pre-Scan Anxiety (Hinglish)", size="sm")
                b2 = gr.Button("2. Masked Distress (Verbal Denial)", size="sm")
            with gr.Row():
                b3 = gr.Button("3. Chemo Weakness & Nausea", size="sm")
                b4 = gr.Button("4. Relieved: 'mai bhot khush hu'", size="sm")
            with gr.Row():
                b5 = gr.Button("5. Severe Crisis Alert (Urgent)", size="sm")

        with gr.Column(scale=6):
            gr.Markdown("### 2️⃣ Assessment Results (At a Glance)")
            out_c1 = gr.HTML()
            out_c2 = gr.HTML()
            out_c3 = gr.HTML()

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 3️⃣ Recommended Support & Counselor Action")
            out_reply = gr.Markdown()

    b1.click(lambda: ("Kal mera pehla chemo aur PET-CT scan hai Tata Memorial me, bohot ghabrahat aur nervousness ho rahi hai. Haath kaanp rahe hain.", None), outputs=[text_in, audio_in])
    b2.click(lambda: ("I am completely fine and normal, sab theek hai, there is no problem at all.", None), outputs=[text_in, audio_in])
    b3.click(lambda: ("Teesra chemo cycle ke baad bohot kamzori hai, lagatar ulti ho rahi hai aur chhati me jalan hai.", None), outputs=[text_in, audio_in])
    b4.click(lambda: ("Mai bhot khush hu! Aaj doctor ne bataya ki mera report bilkul normal hai aur cancer remission me hai.", None), outputs=[text_in, audio_in])
    b5.click(lambda: ("Mujhe lagta hai sab khatam ho gaya hai. Itna dard aur bojh bardasht nahi hota, mar jaana chahta hoon ab.", None), outputs=[text_in, audio_in])

    btn.click(fn=run_reviewer_demo, inputs=[text_in, audio_in, lang_sel], outputs=[out_c1, out_c2, out_c3, out_reply])

print("✓ Gradio Reviewer-Friendly Demo Ready! Launching...")
demo.launch(share=True, debug=True)
